In [ ]:
"""
Ablation study for XPASE (cross-paradigm stacking ensemble, Alzheimer's tabular data).

Run in the same folder as Alzheimer_s_Disease_Dataset.csv (e.g. in Colab after unzipping archive.zip):
    python ablation_study.py
Output: ablation_results.csv and a printout in the same format as Table 4.6 of the manuscript.

Setup mirrors your notebooks exactly: 80/20 stratified split (seed 42), StratifiedKFold(5, shuffle, seed 42),
StandardScaler -> StackingClassifier(cv=5, passthrough=True) with the tuned hyperparameters from Untitled6.
SANITY CHECK: the "Full XPASE (tuned)" row should reproduce CV 95.35 +/- 0.84, test 95.35, F1 93.38, AUC 95.97.
Small differences can occur if library versions differ (SVC has no seed in the notebook; it is fixed to 42 here).
"""
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from lightgbm import LGBMClassifier

df = pd.read_csv("/content/alzheimers_disease_data (1).csv").drop(["PatientID", "DoctorInCharge"], axis=1)
X, y = df.drop("Diagnosis", axis=1), df["Diagnosis"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def tuned_learners():
    return {
        "rf":   RandomForestClassifier(n_estimators=200, min_samples_split=10, min_samples_leaf=4,
                                       max_features=0.7, max_depth=None, bootstrap=True, random_state=42),
        "svm":  SVC(probability=True, kernel="linear", gamma="scale", C=0.5, random_state=42),
        "lgbm": LGBMClassifier(n_estimators=675, learning_rate=0.02996333702317894, max_depth=4, num_leaves=59,
                               min_child_samples=20, subsample=0.7546321106699425,
                               colsample_bytree=0.7146566138819641, reg_alpha=0.7961247579908425,
                               reg_lambda=0.01598294168923899, verbosity=-1, random_state=42),
        "mlp":  MLPClassifier(hidden_layer_sizes=(128,), activation="relu", learning_rate_init=0.01,
                              alpha=0.01, max_iter=500, random_state=42),
        "knn":  KNeighborsClassifier(n_neighbors=23, weights="uniform", p=1, metric="minkowski"),
    }

def untuned_learners():   # the "baseline hybrid" of Untitled0
    return {
        "rf":   RandomForestClassifier(n_estimators=300, random_state=42),
        "svm":  SVC(probability=True, C=2, random_state=42),
        "lgbm": LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=7, num_leaves=40,
                               random_state=42, verbosity=-1),
        "mlp":  MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42),
        "knn":  KNeighborsClassifier(n_neighbors=7),
    }

def tuned_meta():
    return LGBMClassifier(n_estimators=258, learning_rate=0.010667942290088928, max_depth=5, num_leaves=29,
                          reg_alpha=0.4808051394044599, reg_lambda=4.94392430297274, verbosity=-1, random_state=42)

def build(learners, drop=(), keep_only=None, passthrough=True, meta=None):
    est = [(k, v) for k, v in learners.items() if k not in drop and (keep_only is None or k in keep_only)]
    final = meta if meta is not None else tuned_meta()
    return Pipeline([("scaler", StandardScaler()),
                     ("model", StackingClassifier(estimators=est, final_estimator=final,
                                                  passthrough=passthrough, cv=5))])

configs = {
    "Full XPASE (tuned)":                         build(tuned_learners()),
    "Without hyperparameter tuning (baseline hybrid)":
        build(untuned_learners(), meta=LGBMClassifier(n_estimators=200, verbosity=-1)),
    "Without Random Forest":                      build(tuned_learners(), drop=("rf",)),
    "Without SVM":                                build(tuned_learners(), drop=("svm",)),
    "Without LightGBM (base learner)":            build(tuned_learners(), drop=("lgbm",)),
    "Without MLP":                                build(tuned_learners(), drop=("mlp",)),
    "Without KNN":                                build(tuned_learners(), drop=("knn",)),
    "Without passthrough features":               build(tuned_learners(), passthrough=False),
    "Logistic-regression meta-learner":           build(tuned_learners(), meta=LogisticRegression(max_iter=1000)),
    "Tree-based learners only (RF + LightGBM)":   build(tuned_learners(), keep_only=("rf", "lgbm")),
}

rows = []
for name, model in configs.items():
    cvs = cross_val_score(model, X_train, y_train, cv=cv, scoring="accuracy", n_jobs=-1)
    model.fit(X_train, y_train)
    pred, prob = model.predict(X_test), model.predict_proba(X_test)[:, 1]
    rows.append({"Configuration": name,
                 "CV acc. (%)": f"{100*cvs.mean():.2f} ± {100*cvs.std():.2f}",
                 "Test acc. (%)": round(100*accuracy_score(y_test, pred), 2),
                 "F1 (%)": round(100*f1_score(y_test, pred), 2),
                 "ROC-AUC (%)": round(100*roc_auc_score(y_test, prob), 2)})
    print(rows[-1])

res = pd.DataFrame(rows)
res.to_csv("ablation_results.csv", index=False)
print("\n", res.to_string(index=False))

{'Configuration': 'Full XPASE (tuned)', 'CV acc. (%)': '95.40 ± 0.77', 'Test acc. (%)': 95.35, 'F1 (%)': 93.38, 'ROC-AUC (%)': np.float64(95.9)}
{'Configuration': 'Without hyperparameter tuning (baseline hybrid)', 'CV acc. (%)': '95.06 ± 1.06', 'Test acc. (%)': 94.88, 'F1 (%)': 92.72, 'ROC-AUC (%)': np.float64(94.74)}
{'Configuration': 'Without Random Forest', 'CV acc. (%)': '94.88 ± 0.79', 'Test acc. (%)': 95.12, 'F1 (%)': 93.02, 'ROC-AUC (%)': np.float64(95.24)}
{'Configuration': 'Without SVM', 'CV acc. (%)': '95.40 ± 0.77', 'Test acc. (%)': 95.35, 'F1 (%)': 93.38, 'ROC-AUC (%)': np.float64(95.68)}
{'Configuration': 'Without LightGBM (base learner)', 'CV acc. (%)': '95.35 ± 0.84', 'Test acc. (%)': 95.12, 'F1 (%)': 93.02, 'ROC-AUC (%)': np.float64(95.58)}
{'Configuration': 'Without MLP', 'CV acc. (%)': '95.35 ± 0.84', 'Test acc. (%)': 95.35, 'F1 (%)': 93.38, 'ROC-AUC (%)': np.float64(95.77)}
{'Configuration': 'Without KNN', 'CV acc. (%)': '95.35 ± 0.84', 'Test acc. (%)': 95.35, 'F1 (%